In [0]:
%scala
//Load latest snapshot

// import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
// import com.tomtom.addressing.bulk.commons.model.LayerVersions
// import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
// import com.tomtom.addressing.bulk.commons.config.ConfigLoader
// import org.apache.sedona.spark.SedonaContext
// import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
// import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo

// import org.apache.spark.sql.{Dataset, SparkSession}
// import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.openmaps.databricks.common.spark.SparkHelper
// import com.tomtom.addressing.bulk.commons.config.ConfigLoader
// import com.tomtom.addressing.bulk.scala.load.{LoadFreshSnapshotData, LoadDataFromMcr}
// import com.databricks.dbutils_v1.DBUtilsHolder
// import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.openmaps.databricks.common.config.Settings

val mapper = new ObjectMapper()
    mapper.configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
      .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
      .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD" //dbutils.widgets.get("environment").toUpperCase()
// println(s"Environment Name $env")
// val latestRevision = LoadSnapshotInfo.getLatestRevisionId(14533)
// print("latest revision: ", latestRevision)
// // val database = "data_recovery"
// val versionsBuilder = LayerVersions.builder()
// versionsBuilder.layer(14533, latestRevision.get)
// val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
// DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

// val config = SourceConfigLoader.getConfig

// val deltaConfig = config.getTableMapping.getOrDefault(env,"preprocess_dev.source_data")
// val Array(databaseName, tableName) = deltaConfig.split("\\.")
// println(s"Database Name: $databaseName")

val environment = "APT_MERONE_PROD"

implicit val sparky = spark

implicit val settings: Settings = Settings.forEnvironment(environment)


// implicit val envConfig = ConfigLoader.forEnvironment(env.toLowerCase)
// SparkHelper.init("relocation_dc_output")
SparkHelper.init("relocation_dc_output", "relocation_dc_input")

In [0]:
%scala
import com.tomtom.orbis.poi.aqua.pipeline.routingpoint.repository.AddressPointMessageRepository

val addressPointMessageRepository: AddressPointMessageRepository = new AddressPointMessageRepository()
val addressPointMessages = addressPointMessageRepository
      .readAll()

display(addressPointMessages)

In [0]:
deltaChanges = spark.read.format("delta").table("relocation_dc_output.address_location_revert_delta_changes")

display(deltaChanges)

In [0]:
# Write to JSON string using Spark (memory efficient)
# df = spark.read.format("delta").table("relocation_dc_output.all_jobs_address_point_messages")
df = spark.read.format("delta").table("dc_output.all_jobs_address_point_messages")

# Convert to JSON using Spark's built-in toJSON()
json_rdd = df.toJSON()

# Print first N records
for record in json_rdd.take(10):  # Adjust number as needed
    import json
    print(json.dumps(json.loads(record), indent=2))